In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA  # para reduzir dimensoes dos embeddings se necessario
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")  # limpa warnings comuns

# Carrega e executa o phase1.ipynb para disponibilizar os DataFrames no ambiente atual
%run ./phase1.ipynb

# Agora estes DataFrames ficam disponiveis no phase2
df_profiles, df_pairs, df_influencers

In [ ]:
# ========================= Célula 2: embeddings textuais =========================
# Objetivo: transformar texto livre em números que o modelo consiga usar.

# Modelo pré-treinado de linguagem (multilíngue, leve e rápido).
# Cada texto será convertido num vetor de 384 números.
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modelo SentenceTransformer carregado.")

def get_text_embedding(text_series):
    # Em ML não podemos passar NaN/vazio para o encoder.
    # Substituímos por frase neutra para evitar erro e manter o tamanho do dataset.
    text_series = text_series.fillna("No description available").replace("", "No description available")
    
    # Converte lista de textos em vetores numéricos (embeddings).
    # batch_size melhora performance em datasets maiores.
    embeddings = model.encode(text_series.tolist(), show_progress_bar=True, batch_size=32)
    
    # Retorna DataFrame com mesmo índice original (importante para alinhamento de linhas).
    return pd.DataFrame(embeddings, index=text_series.index)

# Perfis: junta headline + about para capturar contexto profissional numa única string.
df_profiles['profile_text'] = df_profiles['headline'].fillna("") + " " + df_profiles['about'].fillna("")

# Gera embeddings para cada perfil.
profile_embeddings = get_text_embedding(df_profiles['profile_text'])
profile_embeddings.columns = [f"emb_prof_{i}" for i in range(profile_embeddings.shape[1])]

# Concatena embeddings ao DataFrame original (colunas novas numéricas).
df_profiles = pd.concat([df_profiles, profile_embeddings], axis=1)

# Posts: usa conteúdo do post como fonte textual principal.
post_embeddings = get_text_embedding(df_influencers['content'])
post_embeddings.columns = [f"emb_post_{i}" for i in range(post_embeddings.shape[1])]
df_influencers = pd.concat([df_influencers, post_embeddings], axis=1)

print("Embeddings gerados:")
print(" - Perfis:", profile_embeddings.shape)
print(" - Posts:", post_embeddings.shape)

In [ ]:
# ========================= Célula 3: pré-processamento tabular =========================
# Objetivo: padronizar variáveis numéricas e codificar categorias em formato numérico.

# Colunas numéricas (serão normalizadas para média 0 e desvio padrão 1).
# Isso ajuda muitos modelos a aprender melhor (especialmente os sensíveis à escala).
num_features_profile = ['num_skills', 'num_experience', 'num_education', 'about_length',
                        'headline_length', 'connections', 'years_experience']
num_features_post = ['followers', 'num_hashtags', 'reactions', 'comments', 'time_spent']

# Colunas categóricas (texto categórico -> one-hot encoding).
cat_features_profile = ['seniority_level', 'industry']
cat_features_post = ['media_type']

# Pipeline de perfis:
# - 'num': aplica StandardScaler nas numéricas
# - 'cat': aplica OneHotEncoder nas categóricas
# - remainder='passthrough': mantém as restantes colunas (ex.: embeddings já numéricos)
preprocessor_profile = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_profile),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features_profile)
    ],
    remainder='passthrough'
)

# Pipeline equivalente para posts.
preprocessor_post = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_post),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features_post)
    ],
    remainder='passthrough'
)

print("Pipelines de pré-processamento criados.")

In [ ]:
# ========================= Célula 4: separar X/y, transformar e fazer split =========================
# Objetivo: remover colunas que não entram no modelo, criar target e dividir treino/val/teste.

# Remove target + IDs + textos brutos (porque texto já virou embedding).
drop_cols_profile = [
    'profile_discoverability_score', 'avg_compatibility_score', 'profile_id', 'name', 'email',
    'headline', 'about', 'skills', 'experience', 'education', 'goals', 'needs', 'can_offer',
    'profile_text', 'remote_preference', 'source'
]

X_profile = df_profiles.drop(columns=drop_cols_profile, errors='ignore')  # features de entrada
y_profile = df_profiles['profile_discoverability_score']                   # variável alvo (o que prever)

drop_cols_post = [
    'post_discoverability_score', 'name', 'headline', 'location', 'about', 'content',
    'content_links', 'media_url', 'hashtags'
]

X_post = df_influencers.drop(columns=drop_cols_post, errors='ignore')
y_post = df_influencers['post_discoverability_score']

# ATENÇÃO (conceito importante de ML):
# Fazer fit_transform ANTES do split pode causar data leakage,
# porque o scaler/encoder "vê" estatísticas de todo o dataset.
# Ideal: split primeiro; depois fit só no treino e transform em val/test.
X_profile_processed = preprocessor_profile.fit_transform(X_profile)
X_post_processed = preprocessor_post.fit_transform(X_post)

print("Dados processados:")
print(" - Perfis shape:", X_profile_processed.shape)
print(" - Posts shape:", X_post_processed.shape)


In [ ]:
# =============================================================================
# 3.4 - Split Treino / Validacao / Teste
# =============================================================================

# Objetivo do split:
# - Treino: dados usados para o modelo "aprender" padroes
# - Validacao: dados usados para ajustar/avaliar durante desenvolvimento
# - Teste: dados finais, usados so no fim para medir performance real

# Proporcao desejada: 70% treino, 15% validacao, 15% teste.
# Fazemos em 2 passos:
# 1) separar 70% treino e 30% temporario
# 2) dividir os 30% temporarios em 2 metades (15% + 15%)

# PERFIS
#  Pega 70% dos dados para treino e o restante (30%) fica temporariamente separado.
X_train_p, X_temp_p, y_train_p, y_temp_p = train_test_split(
    X_profile_processed, y_profile, test_size=0.3, random_state=42
)

# Agora divide os 30% temporarios em 2 metades iguais: 15% validação e 15% teste.
X_val_p, X_test_p, y_val_p, y_test_p = train_test_split(
    X_temp_p, y_temp_p, test_size=0.5, random_state=42
)

# POSTS
# Exatamente a mesma divisão para os posts:
X_train_po, X_temp_po, y_train_po, y_temp_po = train_test_split(
    X_post_processed, y_post, test_size=0.3, random_state=42
)

X_val_po, X_test_po, y_val_po, y_test_po = train_test_split(
    X_temp_po, y_temp_po, test_size=0.5, random_state=42
)

# random_state=42 garante reprodutibilidade:
# ao correr novamente, o split sera sempre o mesmo.

print("Splits criados:")
print(f"Perfis -> Treino: {X_train_p.shape}, Val: {X_val_p.shape}, Test: {X_test_p.shape}")
print(f"Posts   -> Treino: {X_train_po.shape}, Val: {X_val_po.shape}, Test: {X_test_po.shape}")

In [ ]:
# =============================================================================
# 3.5 – Salvar versões preparadas (opcional, mas recomendado)
# =============================================================================

# Perfis
pd.DataFrame(X_train_p).to_csv("X_train_profiles.csv", index=False)
pd.DataFrame(y_train_p).to_csv("y_train_profiles.csv", index=False)
# Repete para val/test se quiseres

# Posts
pd.DataFrame(X_train_po).to_csv("X_train_posts.csv", index=False)
pd.DataFrame(y_train_po).to_csv("y_train_posts.csv", index=False)

print("Dados preparados e salvos. Pronto para modelagem!")